In [96]:
# Paquetes y librerías
library(readr)
library(dplyr)
library(ggplot2)
library(viridisLite)
library(here)  # para rutas relativas
# install.packages("alakazam")
library(alakazam)
library(ineq)
library(vegan)
library(tidyr)
library(tibble)

In [97]:
gt_file <- "/Users/catg/Desktop/SOFIAC/Gitsofia/tesisbioinf-sofia/results/immunesim_ground_truth/gt_E_100.tsv"

gt <- read_tsv(gt_file)

clone_counts <- gt %>%
  select(junction, counts)

Rows: 100 Columns: 7
-- Column specification --------------------------------------------------------
Delimiter: "\t"
chr (5): sequence, v_call, d_call, j_call, junction
dbl (2): counts, freqs

i Use `spec()` to retrieve the full column specification for this data.
i Specify the column types or set `show_col_types = FALSE` to quiet this message.


## Números de Hill como métricas unificadas de diversidad

Los números de Hill representan un marco unificado para el cálculo 
de métricas clásicas de diversidad, permitiendo expresar distintas 
medidas de diversidad dentro de una misma escala: el **número efectivo 
de clones**.

Este enfoque permite integrar métricas clásicas como richness, 
Shannon y Simpson dentro de un mismo sistema, facilitando la 
comparación directa entre repertorios clonales.

A partir de la tabla de números de Hill (`rep_hill_numbers`), 
se extraen valores específicos de diversidad correspondientes 
a distintos órdenes de diversidad (`q`). Para cada métrica, 
se filtra la tabla según el valor de `q` y se extrae el valor 
de diversidad (`d`).

### Métricas clásicas representadas por los números de Hill

- **q = 0 → Richness (riqueza clonal)**  
  Representa el número total de clones únicos presentes 
  en el repertorio. No considera la abundancia relativa 
  de los clones.

- **q = 1 → Shannon (exp(H), número efectivo de clones)**  
  Corresponde al exponencial del índice de Shannon clásico, 
  considerando la frecuencia relativa de los clones.

- **q = 2 → Simpson (dominancia clonal)**  
  Da mayor peso a los clones más abundantes, permitiendo 
  evaluar la dominancia dentro del repertorio.

- **q = 3 y q = 4 → Órdenes superiores de Hill**  
  Incrementan progresivamente el peso de los clones dominantes, 
  permitiendo evaluar estructuras de dominancia más marcadas.

Los valores obtenidos para cada orden de diversidad (`q`) 
se utilizan posteriormente para comparar la diversidad 
clonotípica entre repertorios simulados y diferentes 
condiciones experimentales.

In [98]:

hill_numbers <- function(clone_counts){

    q_values <- 0:4

    diversity_values <- sapply(q_values, function(q) {
        calcDiversity(clone_counts$counts, q)
    })

    hill_table <- data.frame(
        q = q_values,
        d = diversity_values
    )

    return(hill_table)
}

# Ejecutar
rep_hill_numbers <- hill_numbers(clone_counts)

rep_hill_numbers

q,d
<int>,<dbl>
0,100.000000
1,2.742619
2,1.733720
3,1.548048
4,1.478001


In [99]:
richness <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 0) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    return(metric_value)
} 

richness(rep_hill_numbers)

[1] 100

In [100]:
 d1 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 1) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    return(metric_value)
} 

d1(rep_hill_numbers)

[1] 2.742619

In [101]:
shannon <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 1) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    
    return(log(metric_value))
} 

shannon(rep_hill_numbers)

[1] 1.008913

In [102]:
 d2 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 2) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    return(metric_value)
} 

d2(rep_hill_numbers)

[1] 1.73372

In [103]:
simpson <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 2) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    
    return(1/(metric_value))
} 

simpson(rep_hill_numbers)

[1] 0.5767943

In [104]:
 d3 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 3) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    return(metric_value)
} 

d3(rep_hill_numbers)

[1] 1.548048

In [105]:
d4 <- function(hill_numbers_df){
    
      
    metric_value <- hill_numbers_df %>%
        dplyr::filter(q == 4) %>%
        dplyr::slice(1) %>%
        dplyr::pull(d)
        
       
    return(metric_value)
} 

d4(rep_hill_numbers)

[1] 1.478001

In [106]:
abundances <- clone_counts$counts

metricas_chao1ace <- data.frame(
  chao1 = estimateR(abundances)["S.chao1"],
  ace   = estimateR(abundances)["S.ACE"]
)

metricas_chao1ace

,chao1,ace
,<dbl>,<dbl>
S.chao1,106.1765,109.974


In [107]:
chao1 <- function(clones_df){

  metricas_chao1 <- as.numeric(
    vegan::estimateR(clones_df$counts)["S.chao1"]
  )

  return(metricas_chao1)
}
rep_chao1 <- chao1(clone_counts)
print(rep_chao1)



[1] 106.1765


In [108]:
ace <- function(clones_df){

  metricas_ace <- as.numeric(
    vegan::estimateR(clones_df$counts)["S.ACE"]
  )

  return(metricas_ace)
}
rep_ace <- ace(clone_counts)
print(rep_ace)

[1] 109.974


In [109]:
library(ineq)

calc_gini <- function(df) {
  ineq::ineq(df$counts, type = "Gini")
}

gini_result <- data.frame(
  gini = calc_gini(clone_counts)
)

print(gini_result)

       gini
1 0.9740292


In [110]:
gini <- function(df){
  ineq::ineq(df$counts, type = "Gini")
}

gini(clone_counts)

[1] 0.9740292

In [111]:
calc_pielou <- function(df) {

  abund <- df$counts
  H <- vegan::diversity(abund, index = "shannon")
  S <- vegan::specnumber(abund)

  H / log(S)
}

calc_pielou(clone_counts)

[1] 0.2190598

In [112]:
calc_basharin <- function(df) {

  abund <- df$counts

  N <- sum(abund)
  S <- vegan::specnumber(abund)

  if (N == 0 || S <= 1) return(0)

  H <- vegan::diversity(abund, index = "shannon")

  basharin <- H + (S - 1) / (2 * N)

  return(basharin)
}
calc_basharin(clone_counts)

[1] 1.009177

In [113]:
d50_fun <- function(counts) {

  counts <- sort(counts, decreasing = TRUE)
  total <- sum(counts)
  cum <- cumsum(counts)

  which(cum >= 0.5 * total)[1]
}
d50_val <- d50_fun(clone_counts$counts)
d50_val

[1] 1

In [114]:
metricas_diversidad <- c(

  # --- Identidad Hill numbers ---
  richness = richness(rep_hill_numbers),
  d1 = d1(rep_hill_numbers),
  shannon = vegan::diversity(clone_counts$counts, "shannon"),
  d2 = d2(rep_hill_numbers),
  simpson = vegan::diversity(clone_counts$counts, "simpson"),
  d3 = d3(rep_hill_numbers),
  d4 = d4(rep_hill_numbers),

  # --- Abundancia / riqueza estimada ---
  chao1 = as.numeric(vegan::estimateR(clone_counts$counts)["S.chao1"]),
  ace   = as.numeric(vegan::estimateR(clone_counts$counts)["S.ACE"]),

  # --- Desigualdad / equidad ---
  gini = ineq::ineq(clone_counts$counts, type = "Gini"),

  pielou = {
    H <- vegan::diversity(clone_counts$counts, "shannon")
    S <- vegan::specnumber(clone_counts$counts)
    H / log(S)
  },

  basharin = {
    abund <- clone_counts$counts
    N <- sum(abund)
    S <- vegan::specnumber(abund)
    H <- vegan::diversity(abund, "shannon")
    H + (S - 1) / (2 * N)
  },

  d50 = {
    counts <- sort(clone_counts$counts, decreasing = TRUE)
    which(cumsum(counts) >= 0.5 * sum(counts))[1]
  }
)

In [115]:
tabla_diversidad <- as.data.frame(t(metricas_diversidad))

tabla_diversidad$sample_id <- "E100seq"
tabla_diversidad$condition <- "E"
tabla_diversidad$size <- 100

tabla_diversidad <- tabla_diversidad %>%
  dplyr::select(sample_id, condition, size, dplyr::everything())

In [116]:
readr::write_tsv(
  tabla_diversidad,
  "/Users/catg/Desktop/SOFIAC/Gitsofia/tesisbioinf-sofia/results/diversity_metrics/B/diversity_E_100seqs.tsv"
)